In [0]:
# =============================================================
# CONTROLLED CREDIT-RISK LANGGRAPH WORKFLOW
#
# Requires : 10_runtime_config + 20_retrieval_tool +
#            30_credit_risk_model_tool + 40_policy_tool
# Globals used: retrieve_financial_evidence(), apply_credit_policy(), pd
# Exposes  : credit_risk_graph (compiled StateGraph)
#
# Flow:
#   validate_request
#        ↓
#   apply_policy  ← document-completeness gate runs first
#        ↓ conditional routing
#   ┌────────────────────┬─────────────────────────────┐
#   │ DOCUMENT_COMPLETION│ ENHANCED / STANDARD         │
#   │ REQUIRED           │ ANALYST_REVIEW              │
#   │ → document_        │ → retrieve_evidence         │
#   │   completion       │   ↓ conditional routing     │
#   │   (no retrieval)   │ ┌────────────┬─────────────┐ │
#   │                    │ │ enhanced   │ standard    │ │
#   └────────────────────┴─┴────────────┴─────────────┘
#        ↓
#   finalize_case
#
# No LLM is used in this version.
# =============================================================

from typing import Any, TypedDict
from langgraph.graph import StateGraph, START, END
import pandas as pd  # already in scope; imported here for clarity


# ---------------------------------------------------------
# 1. Shared graph state
# ---------------------------------------------------------
class CreditRiskState(TypedDict, total=False):
    borrower_id:      str
    question:         str
    evidence:         list[dict[str, Any]]
    policy_result:    dict[str, Any]
    workflow_route:   str
    workflow_message: str
    final_result:     dict[str, Any]


# ---------------------------------------------------------
# 2. Request-validation node
# ---------------------------------------------------------
def validate_request_node(
    state: CreditRiskState,
) -> dict[str, Any]:
    borrower_id = str(state.get("borrower_id", "")).strip().upper()
    question    = str(state.get("question",    "")).strip()
    if not borrower_id.startswith("B"):
        raise ValueError("borrower_id must use B-format, for example B000187.")
    if not question:
        raise ValueError("question must not be empty.")
    return {"borrower_id": borrower_id, "question": question}


# ---------------------------------------------------------
# 3. Evidence-retrieval node
#
# Cross-notebook functions are captured as default arguments so
# they are bound at definition time, not looked up in globals at
# LangGraph invocation time (avoids NameError in Databricks runtime).
# ---------------------------------------------------------
def retrieve_evidence_node(
    state: CreditRiskState,
    _retrieve=retrieve_financial_evidence,
    _pd=pd,
) -> dict[str, Any]:
    evidence_df = _retrieve(
        borrower_id=state["borrower_id"],
        question=state["question"],
        num_results=6,
    )
    if evidence_df.empty:
        raise RuntimeError("No borrower-specific financial evidence was retrieved.")

    evidence_records = []
    for _, row in evidence_df.iterrows():
        page_number = (
            None if _pd.isna(row["page_number"])
            else int(float(row["page_number"]))
        )
        score = (
            None if ("score" not in row or _pd.isna(row["score"]))
            else float(row["score"])
        )
        section = (
            None if _pd.isna(row["current_section"])
            else str(row["current_section"])
        )
        evidence_records.append({
            "chunk_id":        str(row["chunk_id"]),
            "document_id":     str(row["document_id"]),
            "document_type":   str(row["document_type"]),
            "section":         section,
            "page_number":     page_number,
            "content":         str(row["chunk_to_retrieve"]),
            "retrieval_score": score,
        })

    assert all(
        r["page_number"] is not None for r in evidence_records
    ), "A retrieved chunk is missing page_number."

    return {"evidence": evidence_records}


# ---------------------------------------------------------
# 4. Deterministic model + policy node
# ---------------------------------------------------------
def apply_policy_node(
    state: CreditRiskState,
    _policy=apply_credit_policy,
) -> dict[str, Any]:
    policy_result = _policy(state["borrower_id"])
    return {
        "policy_result":  policy_result,
        "workflow_route": policy_result["workflow_route"],
    }


# ---------------------------------------------------------
# 5. Routing functions
# ---------------------------------------------------------
def select_workflow_route(state: CreditRiskState) -> str:
    """After apply_policy: branch on document completeness first."""
    route = state["workflow_route"]
    allowed = {
        "DOCUMENT_COMPLETION_REQUIRED",
        "ENHANCED_ANALYST_REVIEW",
        "STANDARD_ANALYST_REVIEW",
    }
    if route not in allowed:
        raise RuntimeError(f"Unsupported workflow route: {route}")
    return route


def select_review_route(state: CreditRiskState) -> str:
    """After retrieve_evidence: dispatch to the correct review node."""
    return state["workflow_route"]


# ---------------------------------------------------------
# 6. Route-specific nodes
# ---------------------------------------------------------
def document_completion_node(state: CreditRiskState) -> dict[str, Any]:
    return {"workflow_message": (
        "Required documents or document-integrity issues must "
        "be resolved before the normal credit assessment continues."
    )}


def enhanced_review_node(state: CreditRiskState) -> dict[str, Any]:
    return {"workflow_message": (
        "The case requires enhanced analyst review because one or "
        "more model or financial-risk rules were triggered."
    )}


def standard_review_node(state: CreditRiskState) -> dict[str, Any]:
    return {"workflow_message": (
        "The case follows the standard analyst-review workflow. "
        "No enhanced-review rule was triggered."
    )}


# ---------------------------------------------------------
# 7. Final structured-result node
# ---------------------------------------------------------
def finalize_case_node(state: CreditRiskState) -> dict[str, Any]:
    policy_result = state["policy_result"]
    model_result  = policy_result["model_result"]

    return {
        "final_result": {
            "borrower_id": state["borrower_id"],
            "question":    state["question"],

            "model_assessment": {
                "probability_of_bankruptcy": model_result["probability_of_bankruptcy"],
                "review_threshold":          model_result["review_threshold"],
                "review_required":           model_result["review_required"],
                "decision_support_status":   model_result["decision_support_status"],
                "model_name":                model_result["model_name"],
                "model_alias":               model_result["model_alias"],
                "model_version":             model_result["model_version"],
            },

            "policy_assessment": {
                "policy_version":         policy_result["policy_version"],
                "workflow_route":         policy_result["workflow_route"],
                "route_reason":           policy_result["route_reason"],
                "workflow_message":       state["workflow_message"],
                "triggered_rule_count":   policy_result["triggered_rule_count"],
                "triggered_rules":        policy_result["triggered_rules"],
                "missing_document_types": policy_result["missing_document_types"],
            },

            # Evidence is absent on the DOCUMENT_COMPLETION_REQUIRED path.
            "document_evidence":          state.get("evidence", []),
            "human_review_required":      True,
            "automatic_lending_decision": False,

            "disclaimer": (
                "This prototype supports analyst review. "
                "It does not approve or reject credit and "
                "does not represent NIBC lending policy."
            ),
        }
    }


# ---------------------------------------------------------
# 8. Construct and compile the graph
# ---------------------------------------------------------
graph_builder = StateGraph(CreditRiskState)

graph_builder.add_node("validate_request",   validate_request_node)
graph_builder.add_node("retrieve_evidence",  retrieve_evidence_node)
graph_builder.add_node("apply_policy",        apply_policy_node)
graph_builder.add_node("document_completion", document_completion_node)
graph_builder.add_node("enhanced_review",     enhanced_review_node)
graph_builder.add_node("standard_review",     standard_review_node)
graph_builder.add_node("finalize_case",       finalize_case_node)

graph_builder.add_edge(START,              "validate_request")
graph_builder.add_edge("validate_request", "apply_policy")

graph_builder.add_conditional_edges(
    "apply_policy",
    select_workflow_route,
    {
        "DOCUMENT_COMPLETION_REQUIRED": "document_completion",
        "ENHANCED_ANALYST_REVIEW":      "retrieve_evidence",
        "STANDARD_ANALYST_REVIEW":      "retrieve_evidence",
    },
)

graph_builder.add_conditional_edges(
    "retrieve_evidence",
    select_review_route,
    {
        "ENHANCED_ANALYST_REVIEW": "enhanced_review",
        "STANDARD_ANALYST_REVIEW": "standard_review",
    },
)

graph_builder.add_edge("document_completion", "finalize_case")
graph_builder.add_edge("enhanced_review",      "finalize_case")
graph_builder.add_edge("standard_review",      "finalize_case")
graph_builder.add_edge("finalize_case",        END)

credit_risk_graph = graph_builder.compile()

print("Controlled credit-risk LangGraph compiled.")